In [1]:
# Load libraries
import pandas as pd
import numpy as np

# Load the raw international results data
results = pd.read_csv("../data/raw/results.csv", parse_dates=["date"])

print("Rows and columns:", results.shape)
results.tail()

Rows and columns: (49945, 9)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
49940,2026-10-06,Belize,French Guiana,0,1,CONCACAF Nations League,Belmopan,Belize,False
49941,2026-10-06,Turks and Caicos Islands,Montserrat,1,4,CONCACAF Nations League,Providenciales,Turks and Caicos Islands,False
49942,2026-10-06,Antigua and Barbuda,Aruba,3,2,CONCACAF Nations League,North Sound,Antigua and Barbuda,False
49943,2026-10-06,Saint Martin,Bahamas,6,0,CONCACAF Nations League,Saint Croix,United States Virgin Islands,True
49944,2026-10-06,Saudi Arabia,United Arab Emirates,2,0,Gulf Cup,Jeddah,Saudi Arabia,False


In [2]:
# Keep only the 2026 FIFA World Cup matches (these give us the target: goals)
wc = results[(results["tournament"] == "FIFA World Cup") &
             (results["date"] >= "2026-06-11")].copy()
wc = wc.sort_values("date").reset_index(drop=True)

print("World Cup 2026 matches:", len(wc))
print("First match:", wc["date"].min().date(), "| Last match:", wc["date"].max().date())
print("Number of teams:", pd.concat([wc["home_team"], wc["away_team"]]).nunique())
wc.head()

World Cup 2026 matches: 104
First match: 2026-06-11 | Last match: 2026-07-19
Number of teams: 48


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,2026-06-11,Mexico,South Africa,2,0,FIFA World Cup,Mexico City,Mexico,False
1,2026-06-11,South Korea,Czech Republic,2,1,FIFA World Cup,Zapopan,Mexico,True
2,2026-06-12,Canada,Bosnia and Herzegovina,1,1,FIFA World Cup,Toronto,Canada,False
3,2026-06-12,United States,Paraguay,4,1,FIFA World Cup,Inglewood,United States,False
4,2026-06-13,Qatar,Switzerland,1,1,FIFA World Cup,Santa Clara,United States,True


In [3]:
# Give each World Cup match an ID number (1 to 104, in date order)
wc["match_id"] = range(1, len(wc) + 1)

# One row from the home team's point of view
side_a = pd.DataFrame({
    "match_id": wc["match_id"],
    "date": wc["date"],
    "team": wc["home_team"],
    "opponent": wc["away_team"],
    "goals": wc["home_score"],      # TARGET: goals this team scored
})

# One row from the away team's point of view
side_b = pd.DataFrame({
    "match_id": wc["match_id"],
    "date": wc["date"],
    "team": wc["away_team"],
    "opponent": wc["home_team"],
    "goals": wc["away_score"],
})

# Stack them together: 104 + 104 = 208 rows
df = pd.concat([side_a, side_b]).sort_values("match_id").reset_index(drop=True)

# Checks
print("Rows and columns:", df.shape)
print("Rows per match:", df.groupby("match_id").size().unique())
print("Total goals in df:", df["goals"].sum(),
      "| Total goals in wc:", wc["home_score"].sum() + wc["away_score"].sum())
df.head(6)

Rows and columns: (208, 5)
Rows per match: [2]
Total goals in df: 308 | Total goals in wc: 308


,match_id,date,team,opponent,goals
0,1,2026-06-11,Mexico,South Africa,2
1,1,2026-06-11,South Africa,Mexico,0
2,2,2026-06-11,South Korea,Czech Republic,2
3,2,2026-06-11,Czech Republic,South Korea,1
4,3,2026-06-12,Canada,Bosnia and Herzegovina,1
5,3,2026-06-12,Bosnia and Herzegovina,Canada,1


In [4]:
# Only matches BEFORE the World Cup started (prevents data leakage)
WC_START = pd.Timestamp("2026-06-11")
before = results[results["date"] < WC_START].dropna(subset=["home_score", "away_score"])

# Write every past match twice: once for each team
home_view = pd.DataFrame({
    "date": before["date"], "team": before["home_team"], "opponent": before["away_team"],
    "scored": before["home_score"], "conceded": before["away_score"],
    "tournament": before["tournament"],
})
away_view = pd.DataFrame({
    "date": before["date"], "team": before["away_team"], "opponent": before["home_team"],
    "scored": before["away_score"], "conceded": before["home_score"],
    "tournament": before["tournament"],
})
history = pd.concat([home_view, away_view]).sort_values("date").reset_index(drop=True)

print("History rows:", history.shape)
print("Latest date in history:", history["date"].max().date())

History rows: (98950, 6)
Latest date in history: 2026-06-10


In [5]:
# Each team's last 10 matches before the World Cup
last10 = history.groupby("team").tail(10)
form10 = last10.groupby("team").agg(
    goals_pg_last10=("scored", "mean"),
    conceded_pg_last10=("conceded", "mean"),
    n_matches=("scored", "size"),
)

# Variable 3 (shared): this team's goals per game, last 10
df["team_goals_pg_last10"] = df["team"].map(form10["goals_pg_last10"])
# Variable 5 (unique): the OPPONENT's goals conceded per game, last 10
df["opp_conceded_pg10"] = df["opponent"].map(form10["conceded_pg_last10"])

# Checks
wc_teams = df["team"].unique()
print("Teams with fewer than 10 past matches:",
      list(form10.loc[wc_teams].query("n_matches < 10").index))
print("Missing values:", df[["team_goals_pg_last10", "opp_conceded_pg10"]].isna().sum().to_dict())
df.head(4)

Teams with fewer than 10 past matches: []
Missing values: {'team_goals_pg_last10': 0, 'opp_conceded_pg10': 0}


,match_id,date,team,opponent,goals,team_goals_pg_last10,opp_conceded_pg10
0,1,2026-06-11,Mexico,South Africa,2,1.6,1.0
1,1,2026-06-11,South Africa,Mexico,0,1.5,0.4
2,2,2026-06-11,South Korea,Czech Republic,2,1.5,0.9
3,2,2026-06-11,Czech Republic,South Korea,1,2.2,1.2


In [6]:
# Put every team's past matches and World Cup matches on one timeline
timeline = pd.concat([history[["date", "team"]], df[["date", "team"]]])
timeline = timeline.drop_duplicates().sort_values(["team", "date"])

# Days since each team's previous match
timeline["prev_date"] = timeline.groupby("team")["date"].shift(1)
timeline["rest_days"] = (timeline["date"] - timeline["prev_date"]).dt.days

# Variable 4 (shared): attach rest days to the World Cup rows
df = df.merge(timeline[["team", "date", "rest_days"]], on=["team", "date"], how="left")

# Checks
print("Rows:", len(df), "| Missing rest_days:", df["rest_days"].isna().sum())
print(df["rest_days"].describe().round(1))
print("\nLongest rest before a match:")
print(df.nlargest(5, "rest_days")[["team", "date", "rest_days"]])


Rows: 208 | Missing rest_days: 0
count    208.0
mean       6.3
std        5.2
min        3.0
25%        5.0
50%        6.0
75%        6.2
max       76.0
Name: rest_days, dtype: float64

Longest rest before a match:
       team       date  rest_days
24  Uruguay 2026-06-15       76.0
38  Austria 2026-06-16       15.0
46    Ghana 2026-06-17       15.0
20    Japan 2026-06-14       14.0
31     Iran 2026-06-15       11.0


In [7]:
# Matches since the 2022 World Cup final (19 Dec 2022 to 10 Jun 2026)
since22 = history[history["date"] >= "2022-12-19"]

# Variable 8 (unique): share of this team's matches where it scored 0
fail_rate = since22.groupby("team")["scored"].apply(lambda s: (s == 0).mean())
df["team_fail_to_score_rate"] = df["team"].map(fail_rate)

# Variable 6 (unique): opponent's goals per game in competitive matches (no friendlies)
competitive = since22[since22["tournament"] != "Friendly"]
gpg_comp = competitive.groupby("team")["scored"].mean()
df["opp_gpg_competitive"] = df["opponent"].map(gpg_comp)

# Variable 7 (unique): number of World Cups the opponent played from 1998 to 2022
past_wc = history[(history["tournament"] == "FIFA World Cup") & (history["date"].dt.year >= 1998)]
wc_apps = past_wc.groupby("team")["date"].apply(lambda d: d.dt.year.nunique())
df["opp_wc_apps_since1998"] = df["opponent"].map(wc_apps).fillna(0).astype(int)

# Checks
cols = ["team_fail_to_score_rate", "opp_gpg_competitive", "opp_wc_apps_since1998"]
print("Missing values:", df[cols].isna().sum().to_dict())
print("World Cup appearances range:", df["opp_wc_apps_since1998"].min(), "to", df["opp_wc_apps_since1998"].max())
print("Matches since 2022 per team (smallest):", since22.groupby("team").size().loc[wc_teams].min())
df[["team", "opponent"] + cols].head(4)

Missing values: {'team_fail_to_score_rate': 0, 'opp_gpg_competitive': 0, 'opp_wc_apps_since1998': 0}
World Cup appearances range: 0 to 7
Matches since 2022 per team (smallest): 33


,team,opponent,team_fail_to_score_rate,opp_gpg_competitive,opp_wc_apps_since1998
0,Mexico,South Africa,0.245614,1.621622,3
1,South Africa,Mexico,0.285714,1.555556,7
2,South Korea,Czech Republic,0.170732,1.703704,1
3,Czech Republic,South Korea,0.108108,2.318182,7
